In [32]:
import sys, os
sys.path.append(os.path.abspath(".."))   # sys.path est la liste des dossiers dans lesquels Python va chercher les modules

import torch
import torch.nn as nn
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import clear_output
import random
from torch.utils.data import Dataset, DataLoader
import os
from glob import glob
from PIL import Image
import json
from utils import moving_average

In [33]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

# Choisir le GPU et vérifier
torch.cuda.set_device(0)
print(torch.cuda.current_device())  # doit afficher le numéro choisi dans torch.cuda.set_device()
print(torch.cuda.get_device_name(torch.cuda.current_device()))

Using device: cuda
0
Tesla V100S-PCIE-32GB


In [34]:
batch_size = 512
num_epochs = 10
num_attributs = 60
crop_size = 32
learning_rate = 3e-4

In [35]:
class ImageDataset(Dataset):

    def __init__(self, split, transform=None):
        self.transform = transform

        # get the dictionnary with the crop's infos
        with open(f'../data/dataset_crops/{split}_{crop_size}x{crop_size}_data.json', 'r', encoding='utf-8') as f:
            dico = json.load(f)

        # get all image paths, attributs and classes
        self.data = []

        for path in glob(os.path.join(f"../data/dataset_crops/{split}", "**", "*.png"), recursive=True):
            filename = os.path.basename(path)      # "crop_i.png"
            crop_name = os.path.splitext(filename)[0]   # "crop_i"

            attributs = dico[crop_name]["attributs"]
            damage = dico[crop_name]["niveau de brûlé"]

            self.data.append((path, attributs, damage))


    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):
        img_path, attribut, label = self.data[idx]

        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        attribut = torch.tensor(attribut, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.int64)

        return img, attribut, label


# transformations ajoutées pour faire de la data augmentation autrement il y a de l'overfitting

def random_90_rotation(img):
    angles = [0, 90, 180, 270]
    return transforms.functional.rotate(img, angle=random.choice(angles))

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.Lambda(random_90_rotation),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = ImageDataset(split="train", transform=transform)
val_dataset = ImageDataset(split="val", transform=transform)

In [50]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

train_iterator = iter(train_loader)
val_iterator = iter(val_loader)

In [51]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights   # Import des poids officiels

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)   # Charger un modèle préentraîné

# remplacer la dernière couche pour prédire les attributs
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_attributs)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters in the model: {total_params/1e6:.6f} million")

Total parameters in the model: 11.207292 million


In [52]:
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, fused=True)
criterion = nn.CrossEntropyLoss()

train_loss_list, val_loss_list = [], []
train_acc_list, val_acc_list = [], []

In [75]:
def get_accuracy(split, iterator):
    if split == "train":
        loader = train_loader
    elif split == "val":
        loader = val_loader
    else:
        print("split non définit")
        return None

    model.eval()
    with torch.no_grad():
        try:
            imgs, attributs, _ = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            imgs, attributs, _ = next(iterator)
        imgs, attributs = imgs.to(device), attributs.to(device)
        attributs_preds = model(imgs)
        probs = torch.sigmoid(attributs_preds)
        preds = (probs > 0.5).to(torch.int64)
        n_correct = (preds.view(-1) == attributs.view(-1)).float().sum().item()
        n_total = attributs.shape[0] * num_attributs
    accuracy = n_correct / n_total
    model.train()

    return accuracy, iterator

In [ ]:
for epoch in range(num_epochs):
    for i, (imgs, attributs, _) in tqdm(enumerate(train_loader), total=len(train_loader)):

        imgs, attributs = imgs.to(device), attributs.to(device)

        attributs_preds = model(imgs)
        loss = criterion(attributs_preds, attributs)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()



        if i % 10 == 0:
            train_loss_list.append(loss.item())

            # get val loss
            model.eval()
            with torch.no_grad():
                try:
                    imgs, attributs, _ = next(val_iterator)
                except StopIteration:
                    val_iterator = iter(val_loader)
                    imgs, attributs, _ = next(val_iterator)
                imgs, attributs = imgs.to(device), attributs.to(device)
                attributs_preds = model(imgs)
                val_loss = criterion(attributs_preds, attributs)
                val_loss_list.append(val_loss.item())
            model.train()

            # get accuracy
            train_acc, train_iterator = get_accuracy("train", train_iterator)
            val_acc, val_iterator = get_accuracy("val", val_iterator)
            train_acc_list.append(train_acc)
            val_acc_list.append(val_acc)



    train_loss_list_smoothed = moving_average(train_loss_list)
    val_loss_list_smoothed = moving_average(val_loss_list)
    train_acc_list_smoothed = moving_average(train_acc_list)
    val_acc_list_smoothed = moving_average(val_acc_list)

    clear_output(wait=True)  # efface l'ancien plot

    print(f"moyenne sur les 100 dernières valeurs:   train_acc: {torch.tensor([train_acc_list_smoothed[-100:]]).mean()}   val_acc: {torch.tensor([val_acc_list_smoothed[-100:]]).mean()}")

    plt.figure(figsize=(20, 10))  # plus grand pour 4 sous-plots

    # 1er plot (train+val loss, full)
    plt.subplot(2, 2, 1)
    plt.plot(train_loss_list, c="blue", alpha=0.3)
    plt.plot(train_loss_list_smoothed, c="blue", linestyle='--')
    plt.plot(val_loss_list, c="red", alpha=0.3)
    plt.plot(val_loss_list_smoothed, c="red", linestyle='--')
    plt.title("Train & Val Loss (full)")

    # 2e plot (train+val acc, full)
    plt.subplot(2, 2, 2)
    plt.plot(train_acc_list, c="blue", alpha=0.3)
    plt.plot(train_acc_list_smoothed, c="blue", linestyle='--')
    plt.plot(val_acc_list, c="red", alpha=0.3)
    plt.plot(val_acc_list_smoothed, c="red", linestyle='--')
    plt.title("Train & Val Accuracy (full)")

    # 3e plot (train+val loss, from index 30)
    plt.subplot(2, 2, 3)
    plt.plot(train_loss_list[30:], c="blue", alpha=0.3)
    plt.plot(train_loss_list_smoothed[30:], c="blue", linestyle='--')
    plt.plot(val_loss_list[30:], c="red", alpha=0.3)
    plt.plot(val_loss_list_smoothed[30:], c="red", linestyle='--')
    plt.title("Train & Val Loss (from idx 30)")

    # 4e plot (train+val acc, from index 30)
    plt.subplot(2, 2, 4)
    plt.plot(train_acc_list[30:], c="blue", alpha=0.3)
    plt.plot(train_acc_list_smoothed[30:], c="blue", linestyle='--')
    plt.plot(val_acc_list[30:], c="red", alpha=0.3)
    plt.plot(val_acc_list_smoothed[30:], c="red", linestyle='--')
    plt.title("Train & Val Accuracy (from idx 30)")

    plt.tight_layout()  # ajuste l'espacement
    plt.show()


  8%|▊         | 75/889 [00:12<02:04,  6.55it/s]

In [ ]:
# # sauvegarder les poids du modèle et les paramètres d'entraînement

# torch.save({
#     'epoch': epoch+1,
#     'model_state_dict': model.state_dict(),
#     'optimizer_state_dict': optimizer.state_dict(),
#     'loss': torch.tensor([train_acc_list_smoothed[-100:]]).mean(),
# }, f"weights/regression_{crop_size}x{crop_size}_checkpoint10.pth")